# Assignment 25 : Prompting & LangChain Chains

In [2]:
from langchain_core.prompts import PromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv

In [3]:
load_dotenv()

True

# PART 1 - Prompt Template

## Task 1: PromptTemplate

In [5]:
question = input('Please type in your Question')

In [6]:
prompt = PromptTemplate(template='You are a Phd holder person Answer the following Question : What is your name and what do u do ?')

In [7]:
model = ChatGoogleGenerativeAI(model='gemini-3.5-flash')

In [8]:
dynamic_prompt = PromptTemplate(template='you are a Phd holder person Answer the following Question {Question}',input_variables=['Question'])

In [9]:
finalprompt = dynamic_prompt.invoke({'Question': question})

In [13]:
Response = model.invoke('Who are you ?')

In [14]:
Response.text

'I am Gemini, a large language model created by Google. I am an AI assistant designed to help answer questions, generate text, write code, and assist with a wide variety of tasks. \n\nHow can I help you today?'

## Task 2 : ChatPromptTemplate & Message Template

In [15]:
from langchain_core.prompts import ChatPromptTemplate

In [16]:
role = input()
query = input()

In [17]:
chat_temp = ChatPromptTemplate.from_messages([('system','''You are a Senior {role} with 1+ decade of experiance'''),('human','Help me understand this Query {query}')])

In [18]:
prompt = chat_temp.invoke({'role':role,'query':query})

In [19]:
result = model.invoke(prompt)

In [20]:
print(result.text)

As a Senior Generative AI Architect who has watched this field evolve from simple statistical NLP to the current era of Foundation Models, I get this question a lot. 

While these two terms are often used interchangeably in marketing hype, in AI engineering, they represent two different things: **the Entity (the "What")** versus **the Paradigm (the "How").**

Here is a breakdown of the differences, the architecture, and why this distinction matters for the future of enterprise software.

---

### 1. The Executive Summary
*   **An AI Agent** is a **noun**. It is a specific software entity designed to achieve a goal by perceiving its environment, making decisions, and taking actions using tools.
*   **Agentic AI** is an **adjective/paradigm**. It refers to a class of AI systems, design patterns, and behaviors characterized by autonomy, proactivity, reasoning, and iterative execution.

Think of it this way: **An AI Agent is a self-driving car. Agentic AI is the technology and philosophy o

# PART 2: Strectured Output using Pydantic

## Task 3: Pydantic Strecture output

In [22]:
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel
from langchain_openai import ChatOpenAI

In [23]:
class Answer(BaseModel):
    answer : str
    confidence : float
    source : str

In [24]:
parser = PydanticOutputParser(pydantic_object=Answer)

In [25]:
gemini = ChatGoogleGenerativeAI(model='gemini-3.5-flash')

In [26]:
prompt = PromptTemplate(template='''You are a Senior {role}, answer the following Question {Question} in this {format}''',
                        input_variables=['role','Question'],partial_variables={'format':parser.get_format_instructions})

In [27]:
final_prompt = prompt.invoke({'role':'AI Engineer','Question':'which field is best Data Engineering or Gen AI Engineer'})

In [28]:
response = gemini.invoke(final_prompt)

In [29]:
gpt = ChatOpenAI(model='gpt-4o')

In [40]:
response.content[0]

{'type': 'text',
 'text': '```json\n{\n  "answer": "As a Senior AI Engineer, the choice between Data Engineering and Gen AI Engineering depends on your career objectives, as both are highly valuable and deeply interdependent. Data Engineering is the indispensable bedrock of the entire AI lifecycle. Without robust pipelines, data lakes, and governance, AI models cannot function (\'garbage in, garbage out\'). It offers exceptional job stability, mature tooling (e.g., Spark, Airflow, Snowflake), and a massive job market. Gen AI Engineering is the cutting-edge frontier, focusing on Large Language Models (LLMs), Retrieval-Augmented Generation (RAG), vector databases, and agentic workflows. It offers rapid innovation, high-impact business visibility, and premium compensation, though it requires adapting to highly volatile tooling and frameworks. If you prefer building highly scalable, resilient, and foundational data infrastructure, Data Engineering is best. If you prefer rapid prototyping, 

In [41]:
result = gpt.invoke(final_prompt)

In [42]:
print(result.content)

```json
{
  "answer": "Both Data Engineering and Gen AI Engineering have their unique advantages and opportunities. Data Engineering is crucial for processing and managing large volumes of data, which is the backbone for any AI or machine learning model, including generative AI. On the other hand, Gen AI Engineering offers exciting opportunities to work on cutting-edge AI models that generate new and creative content. The best field depends on your career goals and interests. If you're interested in data infrastructure and pipeline building, Data Engineering may be more suitable. If you're passionate about AI model development and creativity, Gen AI Engineering might be a better fit.",
  "confidence": 0.85,
  "source": "AI Career Guides 2023"
}
```


# Part 3 - Chains in langchain

## Task 5: Simple Chain

In [43]:
chain = prompt | gpt | parser

In [44]:
response = chain.invoke({'role':'AI Engineer','Question':'Give me a joke on AI Engineer'})

In [45]:
print(response)

answer="Why don't AI engineers need to worry about bugs? Because they have plenty of 'byte' repellents!" confidence=0.95 source='Traditional tech humor'


## Task 6: Conditional Chain

In [46]:
age = input('Enter your Age')

In [73]:
from langchain_core.runnables import RunnableBranch,RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from typing import Literal
from pydantic import BaseModel,Field

In [74]:
class Sentiment(BaseModel):
    Fact : Literal['True','False'] = Field(description='return only True or False')

In [75]:
parser = PydanticOutputParser(pydantic_object=Sentiment)

In [76]:
conditional_prompt = PromptTemplate(template='You are a Fact Checker, check this {fact} and in {format}',
                        input_variables=['fact'],partial_variables={'format':parser.get_format_instructions})

In [77]:
# fact_prompt = conditional_prompt.invoke({'fact':'Modi is the PM of india'})

In [78]:
from langchain_groq import ChatGroq

In [89]:
chain_gemini = ChatGroq(model='llama-3.1-8b-instant')
chain_gpt = ChatOpenAI(model='gpt-4o')

In [90]:
# res = chain_gemini.invoke(fact_prompt)

In [91]:
# res.content

In [92]:
chain_1 = conditional_prompt | chain_gemini | parser

In [93]:
true_prompt = PromptTemplate(template='Generate the Reason why its {fact}',input_variables=['fact'])
false_prompt = PromptTemplate(template='Generate the Reason why its {fact}',input_variables=['fact'])

In [94]:
strParser = StrOutputParser()

In [95]:
chain_2 = true_prompt | chain_gpt | strParser
chain_3 = true_prompt | chain_gpt | strParser

In [96]:
conditional_chain = RunnableBranch((lambda x:x.Fact=='True',chain_2),(lambda x : x.Fact=='False',chain_3),
                                   RunnableLambda(lambda x : "Sorry cant confirm your Fact is True or False"))

In [97]:
final_chain= chain_1 | conditional_chain

In [102]:
result = final_chain.invoke({'fact':"Taj Mahal is located in Agra ?"})

In [103]:
result

'To generate a reason why a statement is considered factually true, you need to present evidence or a logical explanation that supports the claim. Here’s an example to illustrate how to do this:\n\n**Statement:** Honey never spoils.\n\n**Reason why it\'s Fact: \'True\':**\n\n1. **Scientific Properties:** Honey has a unique composition that prevents it from spoiling. Its low water content and high acidity create an inhospitable environment for bacteria and microorganisms. With a pH level between 3.2 and 4.5 and its hygroscopic nature (absorbing moisture), honey can dehydrate bacteria, preventing their growth.\n\n2. **Historical Evidence:** Archaeologists have found pots of honey in ancient Egyptian tombs that are over 3000 years old and still perfectly edible. This historical evidence further supports the idea that honey\'s properties allow it to remain unspoiled over millennia.\n\n3. **Enzymatic Activity:** Bees add an enzyme called glucose oxidase to honey. When honey is sealed and st

In [104]:
print(final_chain.get_graph().draw_ascii())

    +-------------+      
    | PromptInput |      
    +-------------+      
            *            
            *            
            *            
   +----------------+    
   | PromptTemplate |    
   +----------------+    
            *            
            *            
            *            
      +----------+       
      | ChatGroq |       
      +----------+       
            *            
            *            
            *            
+----------------------+ 
| PydanticOutputParser | 
+----------------------+ 
            *            
            *            
            *            
       +--------+        
       | Branch |        
       +--------+        
            *            
            *            
            *            
    +--------------+     
    | BranchOutput |     
    +--------------+     


## Task 7: Parallel Chain

In [105]:
from langchain_core.output_parsers import PydanticOutputParser,StrOutputParser
from langchain_core.runnables import RunnableParallel

In [218]:
par_model_1 = ChatOpenAI(model='gpt-3.5-turbo')
par_model_2 = ChatOpenAI(model='gpt-4o')
par_model_3 = ChatGoogleGenerativeAI(model='gemini-3.1-flash-lite')


In [219]:
strrParser = StrOutputParser()

In [220]:
ans_prompt = PromptTemplate(template='Generate the Answer on the {topic}',input_variables=['topic'])
sum_prompt = PromptTemplate(template='Generate the Summary on the {topic} in bullet points',input_variables=['topic'])
quiz_prompt = PromptTemplate(template='Generate the follow up Questions on the {topic} for practice',input_variables=['topic'])

In [221]:
par_chain_1 = ans_prompt | par_model_1 | strrParser
par_chain_2 = sum_prompt | par_model_2 | strrParser
par_chain_3 = quiz_prompt | par_model_3 | strrParser

In [222]:
parallel_chain = RunnableParallel({'answer':ans_prompt | par_model_1 | strrParser,'summary':par_chain_2,'quiz':par_chain_3})

In [223]:
final_model = ChatGoogleGenerativeAI(model='gemini-3.5-flash')

In [224]:
class QuestionAnswer(BaseModel):
    Question : str =Field(description='Return only the exact Question which is asked ')
    answer : str = Field(description='return Only the Answer')
    summary : str = Field(description='Only Return the summary from the Question asked')
    quiz : list[str] = Field(description='Only Return the 5 follow-up Question You are asking')

pyParser = PydanticOutputParser(pydantic_object=QuestionAnswer)

In [225]:
final_prompt = PromptTemplate(template='Marge the following {answer}, {summary} and {quiz} together and return the result in this {format}',input_variables=['answer','summary','quiz'],partial_variables={'format':pyParser.get_format_instructions})

In [226]:
single_chain = final_prompt | final_model | pyParser

In [227]:
final_chain = parallel_chain | single_chain

In [228]:
parallel_chain_resul= final_chain.invoke({'topic':"10 + 10 = ?"})

In [229]:
print(parallel_chain_resul)

Question='10 + 10 = ?' answer='20' summary="The equation 10 + 10 = 20 is a basic arithmetic problem involving addition, where the '+' symbol signifies combining the two numbers to find their total. Understanding simple addition is fundamental for daily tasks like budgeting and measurement, serves as a stepping stone to advanced mathematics, and is key to developing mental math fluency." quiz=['If 10 + 10 = 20, what is 10 + 11?', 'What is 20 + 20?', 'If you have 20 and you add 10, what is the new total?', '10 + ___ = 20', 'You have 10 red apples and 10 green apples. How many apples do you have in total?']


In [230]:
for i in parallel_chain_resul.quiz:
    print(i,end='\n')

If 10 + 10 = 20, what is 10 + 11?
What is 20 + 20?
If you have 20 and you add 10, what is the new total?
10 + ___ = 20
You have 10 red apples and 10 green apples. How many apples do you have in total?


In [210]:
# print(final_chain.get_graph().draw_ascii())

In [ ]:
# import streamlit as st 

# st.title(" Parallel Chain Model")

# question = st.text_input('Type in your Question ?')



# PART 4 - Runnables & LCEL

## Task 8:  Rubnnables Basics

In [115]:
from langchain_core.runnables import RunnableLambda,RunnableSequence

In [116]:
def Success():
    return "Answer Generated Successfully"

def Model():
    llm = ChatGroq(model='llama-3.1-8b-instant')
    return llm

In [117]:
R1 = RunnableLambda(Success)
R2 = RunnableLambda(Model)

In [118]:
RunnableSequence(R1,R2)

RunnableLambda(Success)
| RunnableLambda(Model)

In [119]:
chain = R1 | R2

## Task 9: LCEL - Based RAG Chain

In [232]:
question="Who are you ?"
llm = ChatGoogleGenerativeAI(model='gemini-3.5-flash')
prompt = PromptTemplate(template='You are an Question Answer AI Assistent, Generate the Answer for this {question}',input_variables=['question'])
parser = StrOutputParser()

In [233]:
chain = prompt | llm | parser

## Task 10: Observations & Insights

### Answer 1 : 
Structured Output are important so we can get the outputs in structured format from the LLMs and save then in a structured format in a Database for easy retrival, search, etc. 

### Answer 2 : 

#### Advantages of LCEL:
#####   1. No need to import runnable, runnable sequence to define the Chain flow
#####   2. just use pipe operator ('|') to run define the chain flow
#####   eg : Chain = prompt | llm | Parser
#####   3. low code 
#####   4. Supports async naturally


### Answer 3 :
Parallel Chain : Use it when you want to perform multiple tasks Simintunatioly
Conditional Chain : Used when a certain chain should be execute Based on conditions or answer we receive. 